In [1]:
import os
print(os.getcwd())

d:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\feature_extraction


## Alternative method using donwsampled PNG by PNG for feature extraction

### Feature extraction function

In [1]:
# !pip install python-dotenv
from dotenv import load_dotenv
import os

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [2]:
import torch
import timm
from PIL import Image
from torchvision import transforms
from huggingface_hub import login

class HOptimusExtractorCrops:
    def __init__(self, device='cuda'):
        # Login to Hugging Face (if needed)
        login(token=hf_token)

        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')

        self.model = timm.create_model(
            "hf-hub:bioptimus/H-optimus-1",
            pretrained=True,
            init_values=1e-5,
            dynamic_img_size=False
        ).to(self.device)
        self.model.eval()


        self.eval_transform = transforms.Compose([
            transforms.Resize((224, 224)),  # Resize each crop to 224x224
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.707223, 0.578729, 0.703617),
                std=(0.211883, 0.230117, 0.177517)
            )
        ])

        self.fivecrop_transform = transforms.Compose([
            transforms.FiveCrop(256),
            transforms.Lambda(lambda crops: torch.stack([self.eval_transform(c) for c in crops]))
        ])

        self.tencrop_transform = transforms.Compose([
            transforms.TenCrop(256),
            transforms.Lambda(lambda crops: torch.stack([self.eval_transform(c) for c in crops]))
        ])

    def extract_fivecrop_features(self, image: Image.Image | str) -> torch.Tensor:
        """
        Extract features from 5 crops of an image.
        Returns a tensor of shape (5, 1536)
        """
        if isinstance(image, str):
            image = Image.open(image).convert("RGB")
        else:
            image = image.convert("RGB")

        crops_tensor = self.fivecrop_transform(image).to(self.device)  # (5, 3, 224, 224)

        with torch.autocast(device_type=self.device.type, dtype=torch.float16):
            with torch.inference_mode():
                features = self.model(crops_tensor)  # (5, 1536)

        return features.cpu()
    
    def extract_tencrop_features(self, image: Image.Image | str) -> torch.Tensor:
        """
        Extract features from 10 crops of an image.
        Returns a tensor of shape (10, 1536)
        """
        if isinstance(image, str):
            image = Image.open(image).convert("RGB")
        else:
            image = image.convert("RGB")

        crops_tensor = self.tencrop_transform(image).to(self.device)  # (10, 3, 224, 224)

        with torch.autocast(device_type=self.device.type, dtype=torch.float16):
            with torch.inference_mode():
                features = self.model(crops_tensor)  # (10, 1536)

        return features.cpu()

## Testing

In [4]:
# FiveCrop feature extraction function
import glob
import os
from tqdm import tqdm
import torch

# Set paths
patch_root = r"D:\Aamir Gulzar\KSA_project2\dataset\test_patch_data"
save_root = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\H_Optimus_1_tencrop_test"

# Initialize extractor
extractor = HOptimusExtractorCrops()

# Loop through all PNG patches
all_patch_paths = glob.glob(os.path.join(patch_root, "*", "*.png"))

for patch_path in tqdm(all_patch_paths):
    try:
        # For FiveCrop feature extraction, use:
        # features = extractor.extract_fivecrop_features(patch_path)  # shape: (5, D)

        # For TenCrop feature extraction, use:
        features = extractor.extract_tencrop_features(patch_path)  # shape: (10, D)

        # Extract subfolder and filename
        relative_path = os.path.relpath(patch_path, patch_root)
        slide_folder, patch_filename = os.path.split(relative_path)
        patch_base = os.path.splitext(patch_filename)[0]  # remove .png

        # Create save path
        save_dir = os.path.join(save_root, slide_folder)
        os.makedirs(save_dir, exist_ok=True)

        save_path = os.path.join(save_dir, patch_base + ".pt")
        torch.save(features, save_path)
    
    except Exception as e:
        print(f"Error processing {patch_path}: {e}")

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\datainsight\.cache\huggingface\token
Login successful


  0%|          | 0/6164 [00:00<?, ?it/s]c:\Users\datainsight\anaconda3\envs\exaonepath\lib\site-packages\timm\layers\attention.py:80: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  x = F.scaled_dot_product_attention(
  0%|          | 10/6164 [00:01<14:02,  7.31it/s]


KeyboardInterrupt: 

## Final Pipeline

In [3]:
import os
import glob
import pandas as pd
import torch
from tqdm import tqdm

def run_full_pipeline_HOptimus_crops(
    patch_data_dir: str,
    extractor,  # Instance of HOptimusExtractorCrops
    non_white_csv: str,
    output_root: str
):
    # Load non-white metadata
    non_white_df = pd.read_csv(non_white_csv)
    valid_patch_names = set(non_white_df['patch_name'].astype(str))

    # Loop over slide directories
    slide_dirs = glob.glob(os.path.join(patch_data_dir, "*"))
    for slide_dir in tqdm(slide_dirs, desc="🧩 Processing slides", unit="slide", position=0):
        slide_name = os.path.basename(slide_dir)
        patch_feat_dir = os.path.join(output_root, slide_name)
        os.makedirs(patch_feat_dir, exist_ok=True)

        # Filter valid non-white patches
        all_patches = glob.glob(os.path.join(slide_dir, "*.png"))
        valid_patch_paths = [p for p in all_patches if os.path.basename(p) in valid_patch_names]

        if not valid_patch_paths:
            tqdm.write(f"  ❌ No valid non-white patches found for {slide_name}")
            continue

        # Skip already processed patches
        existing_feats = set(os.path.splitext(f.name)[0] for f in os.scandir(patch_feat_dir) if f.name.endswith(".pt"))
        to_process = [p for p in valid_patch_paths if os.path.splitext(os.path.basename(p))[0] not in existing_feats]

        if not to_process:
            tqdm.write(f"  ⏭️ Skipped: All {len(valid_patch_paths)} patches already processed for {slide_name}")
            continue

        # Patch-level tqdm bar
        for patch_path in tqdm(to_process, desc=f"🔄 {slide_name}", unit="patch", position=1, leave=False):
            patch_name = os.path.splitext(os.path.basename(patch_path))[0]
            save_path = os.path.join(patch_feat_dir, patch_name + ".pt")

            try:
                features = extractor.extract_tencrop_features(patch_path)  # shape (5, D)
                torch.save(features, save_path)
            except Exception as e:
                tqdm.write(f"  ❌ Failed to process patch {patch_path}: {e}")
                continue

        tqdm.write(f"  ✅ Done: Processed {len(to_process)} new patches (total: {len(valid_patch_paths)}) for {slide_name}")

In [ ]:
extractor = HOptimusExtractorCrops()

run_full_pipeline_HOptimus_crops(
    patch_data_dir=r"D:\Aamir Gulzar\KSA_project2\dataset\patch_data",
    extractor=extractor,
    non_white_csv=r"D:\Aamir Gulzar\KSA_project2\dataset\final_merged_without_white.csv",
    output_root=r"D:\Aamir Gulzar\KSA_project2\dataset\Features\H_Optimus_1_tencrop"
)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\datainsight\.cache\huggingface\token
Login successful


🧩 Processing slides:   0%|          | 1/417 [00:02<14:35,  2.11s/slide]

  ⏭️ Skipped: All 1878 patches already processed for TCGA-3L-AA1B_nonMSIH


🧩 Processing slides:   0%|          | 2/417 [00:06<23:29,  3.40s/slide]

  ⏭️ Skipped: All 4240 patches already processed for TCGA-4N-A93T_nonMSIH


...

  ✅ Done: Processed 3020 new patches (total: 3020) for TCGA-G4-6626_nonMSIH


🧩 Processing slides:  95%|█████████▌| 398/417 [17:36:57<2:10:30, 412.15s/slide]

  ✅ Done: Processed 4823 new patches (total: 4823) for TCGA-G4-6627_nonMSIH


🧩 Processing slides:  96%|█████████▌| 399/417 [17:44:52<2:09:21, 431.18s/slide]

  ✅ Done: Processed 4592 new patches (total: 4592) for TCGA-G4-6628_MSIH


## Check .pt file

In [5]:
import torch
f = torch.load(r"D:\Aamir Gulzar\KSA_project2\dataset\Features\H_Optimus_1_tencrop\TCGA-3L-AA1B_nonMSIH\TCGA-3L-AA1B_nonMSIH_x2608_y14048_patch00000.pt")
print(f.shape)       # Should be torch.Size([5, 1536]) or torch.Size([10, 1536]) depending on fivecrop vs tencrop
print(f.element_size(), f.numel(), f.element_size() * f.numel() / 1024, "KB")

torch.Size([10, 1536])
4 15360 60.0 KB


In [6]:
import torch

def inspect_pt_file(pt_path):
    """Inspect the contents of a PyTorch .pt file"""
    print(f"Inspecting PT file: {pt_path}")
    print("=" * 50)

    data = torch.load(pt_path, map_location='cpu')

    if isinstance(data, dict):
        for key, value in data.items():
            print(f"Key: {key}")
            if isinstance(value, torch.Tensor):
                print(f"  Type: Tensor")
                print(f"  Shape: {tuple(value.shape)}")
                print(f"  Dtype: {value.dtype}")
                print("  First 10 values in each of the 5 rows:")
                for i in range(min(5, value.shape[0])):  # Only first 5 rows
                    row_values = value[i][:10].tolist()
                    print(f"    Row {i+1}: {row_values}")
            else:
                print(f"  Type: {type(value)}")
                print(f"  Value: {value}")
            print()
    elif isinstance(data, torch.Tensor):
        print(f"Single Tensor:")
        print(f"  Shape: {tuple(data.shape)}")
        print(f"  Dtype: {data.dtype}")
        print("  First 10 values in each of the 5 rows:")
        for i in range(min(5, data.shape[0])):
            row_values = data[i][:10].tolist()
            print(f"    Row {i+1}: {row_values}")
    else:
        print("The .pt file is not a dictionary or tensor. Type:", type(data))
        print("Value preview:", str(data)[:1500])  # Print partial value


pt_path = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\H_Optimus_1_tencrop\TCGA-3L-AA1B_nonMSIH\TCGA-3L-AA1B_nonMSIH_x2608_y14048_patch00000.pt"
inspect_pt_file(pt_path)


Inspecting PT file: D:\Aamir Gulzar\KSA_project2\dataset\Features\H_Optimus_1_tencrop\TCGA-3L-AA1B_nonMSIH\TCGA-3L-AA1B_nonMSIH_x2608_y14048_patch00000.pt
Single Tensor:
  Shape: (10, 1536)
  Dtype: torch.float32
  First 10 values in each of the 5 rows:
    Row 1: [0.43928006291389465, -0.00897279568016529, -0.4755344092845917, 0.4581318795681, -1.0974781513214111, -0.7894259691238403, 0.5287072658538818, 0.2983981668949127, -0.4545738697052002, -0.6002600789070129]
    Row 2: [0.0416511669754982, -0.24505604803562164, -0.04426280036568642, -0.5704256296157837, -1.272540807723999, 0.09869521856307983, -0.6195875406265259, 0.9790215492248535, -0.22534270584583282, -0.8458552360534668]
    Row 3: [-0.8229987025260925, -0.003493713913485408, -0.3740840554237366, -0.44870007038116455, 0.05157395079731941, -1.0504001379013062, 0.5062860250473022, 0.14719538390636444, -0.7002995610237122, 0.016718562692403793]
    Row 4: [-0.23178955912590027, -0.18274915218353271, 0.2269112914800644, -0.544

In [9]:
import torch
import os

def run_sanity_checks(tensor: torch.Tensor):
    print("Running sanity checks...\n")

    # 1. Shape
    if tensor.shape == (5, 1536):
        print(f"Shape OK for FiveCrop: {tuple(tensor.shape)}")
    elif tensor.shape == (10, 1536):
        print(f"Shape OK for TenCrop: {tuple(tensor.shape)}")
    else:
        print(f"Shape mismatch: Expected (5, 1536) or (10, 1536), got {tuple(tensor.shape)}")

    # 2. Dtype
    if tensor.dtype not in [torch.float32, torch.float16]:
        print(f"Unexpected dtype: {tensor.dtype}")
    else:
        print(f"Dtype OK: {tensor.dtype}")

    # 3. NaN or Inf
    if not torch.isfinite(tensor).all():
        nan_count = torch.isnan(tensor).sum().item()
        inf_count = torch.isinf(tensor).sum().item()
        print(f"Invalid values found — NaNs: {nan_count}, Infs: {inf_count}")
    else:
        print(f"No NaN or Inf values")

    # 4. Value range
    min_val = tensor.min().item()
    max_val = tensor.max().item()
    print(f"Value range: min={min_val:.4f}, max={max_val:.4f}")
    if abs(min_val) > 1e6 or abs(max_val) > 1e6:
        print("Warning: Value range is unusually large.")

    # 5. L2 norm (per row)
    norms = tensor.norm(p=2, dim=1)
    print(f"L2 Norms (per row): {norms.tolist()}")
    if (norms == 0).any():
        print("Zero vector detected (row with all zeros)")
    else:
        print("All rows have non-zero magnitude")

    # 6. Row consistency (std across rows)
    row_std = torch.std(tensor, dim=1)
    print(f"Row-wise std dev: {row_std.tolist()}")
    if (row_std == 0).any():
        print("Warning: At least one row is constant (no variance)")

    print("\nSanity check completed.")

# Example usage
pt_path = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\H_Optimus_1_tencrop\TCGA-3L-AA1B_nonMSIH\TCGA-3L-AA1B_nonMSIH_x2608_y14048_patch00000.pt"
data = torch.load(pt_path, map_location='cpu')

# If it's a dict with a tensor inside
if isinstance(data, dict):
    for key in data:
        if isinstance(data[key], torch.Tensor) and data[key].shape == (5, 1536):
            run_sanity_checks(data[key])
            break
else:
    run_sanity_checks(data)


Running sanity checks...

Shape OK for TenCrop: (10, 1536)
Dtype OK: torch.float32
No NaN or Inf values
Value range: min=-2.8872, max=2.8594
L2 Norms (per row): [23.30494499206543, 26.800235748291016, 29.622661590576172, 28.305131912231445, 26.38971519470215, 26.991561889648438, 23.154817581176758, 28.464801788330078, 29.924419403076172, 26.16554832458496]
All rows have non-zero magnitude
Row-wise std dev: [0.5947872400283813, 0.6840364336967468, 0.7560691237449646, 0.7224469184875488, 0.6735631823539734, 0.6889222860336304, 0.5909380912780762, 0.7265185713768005, 0.76376873254776, 0.6678422093391418]

Sanity check completed.
